In [22]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('Data/vehicles.csv')

임시로 VIN 형식에 맞는 전처리 진행 후, 20개의 데이터에서만 VIN API 호출을 진행한다.

VIN 형식
- 17자
- 대문자와 숫자의 조합으로만 존재한다.
- 알파벳 중 I, O, Q는 포함하지 않는다.

In [23]:
# Filter VINs by real VIN format:
# - exactly 17 characters
# - uppercase letters/digits only
# - excludes I, O, Q
vin_series = df['VIN'].astype(str).str.strip().str.upper()
df = df[vin_series.str.fullmatch(r'[A-HJ-NPR-Z0-9]{17}', na=False)]


VIN_sample = df["VIN"].head(20).tolist() 
model_sample = df["model"].head(20).tolist()

VIN API는 NHTSA(National Highway Traffic Safety Administration) 미국 고속도로교통안전국의 무료 API이다.
API 호출은 총 50 개의 데이터로 분할 요청되고 각각의 값들을 `vin_api_result.csv` 파일에 저장된다.

VIN API값을 통해 출고가 가격을 분석할 수 있는 정보는 다음과 같다.
- Make (제조사) : 브랜드의 프리미엄(Brand Tax)을 결정한다.
- Model (모델명) : 차량의 세그먼트(소형, 중형, 대형 등)를 확정한다.
- Series (시리즈) : 픽업트럭과 대형 밴에서 체급을 나눈다,
- Trim (등급/트림) : 트림을 통해 옵션 패키지 가격을 추가해 반영한다.
- FuelTypePrimary (주 연료) : 연료마다 가격이 상이한다.
- DriveType (구동 방식) : 2륜 구동에 비해 4륜 구동 옵션이 더 비싸다. 
- EngineCylinders (기통 수) & DisplacementL (배기량) : 트림에서 누락 될 수 있는 옵션들
- TransmissionStyle (변속기 형태) :PDK나 수동 변속기가 자동에 비해 매니아용 스포츠카의 가격 프리미엄이 생긴다.
- Doors (도어 수) : 트럭의 경우, 문이 2개인지 4개인지에 따라 가격이 상이하다.

또한 원본 데이터에서 판매자가 VIN을 조작하여 입력했을 수 있기 때문에 이를 두 model column을 비교하여 일치 여부에 대한 Feature을 추가한다.
단, 여기서 원본 데이터의 경우 판매자가 모델에 시리즈를 포함하여 적을 수도 있고 공백/하이픈, 대소문자가 다 다를 수 있기 때문에 이를 미리 전처리 후 비교해야 한다.

- IsFraud : VIN이 조작되어 있는지 아닌지 판단
- Log : IsFraud일 경우 원본 데이터의 model명

In [ ]:
import re

def check_vin_fraud(origin_data_model, vin_api_result_model):
    if origin_data_model == None or vin_api_result_model == None:
        return True
    
    # normalize text (lowercase, remove space, remove hyphen)
    def normalize_text(text):
        return re.sub(r'[^a-z0-9]', '', str(text).lower())

    normalized_origin_model = normalize_text(origin_data_model)
    normalized_vin_model = normalize_text(vin_api_result_model)
    
    # check if normalized_origin_model is in normalized_vin_model
    if (normalized_origin_model in normalized_vin_model) or (normalized_vin_model in normalized_origin_model):
        return False
    else:
        return True

In [25]:
import requests
import time
from tqdm import tqdm

# NHTSA vPIC(Vehicle Product Information Catalog) API
# Don't Need API KEY
API_URL = "https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVINValuesBatch/"

# We need to get these features from VIN API
FEATURES = ["VIN", "ModelYear", "Make", "Model", "Series", "FuelTypePrimary", "Trim", "DriveType", "EngineCylinders", "DisplacementL", "TransmissionStyle", "Doors", "IsFraud", "Log"]

def fetch_to_csv(VIN_List, model_List, output_file = "vin_api_result.csv"):
    results = []

    # split 50 batch for request
    for i in tqdm(range(0, len(VIN_List), 50)):
        vin_chuck = VIN_List[i:i+50]
        model_chuck = model_List[i:i+50]

        api_results = []

        # fill null value for each feature (init states)
        for v in vin_chuck:
            place_holder = {feat: None for feat in FEATURES}
            place_holder["VIN"] = v
            api_results.append(place_holder)

        # request to VIN API
        try :
            payload = {"format": "json", "data": ";".join(vin_chuck)}
            response = requests.post(API_URL, data=payload, timeout=20)
            response.raise_for_status()

            api_data = response.json().get("Results", [])
            
            # fill api_result with api_data
            for res in api_data:
                target_vin = res.get("VIN")
                for p in api_results:
                    if p["VIN"] == target_vin:
                        for k, v in res.items():
                            if k in FEATURES:
                                p[k] = v

            # Check fraud
            for idx, p in enumerate(api_results):
                Isfraud = check_vin_fraud(model_chuck[idx], p.get("Model"))
                if Isfraud:
                    p["IsFraud"] = True
                    p["Log"] = model_chuck[idx]
                else:
                    p["IsFraud"] = False
                    p["Log"] = None
        
        except Exception as e:
            print(f"\n[Error] {i}번 배치 처리 중 문제 발생 (null로 대체): {e}")

        results.extend(api_results)
        time.sleep(1) # prevent rate limit

    df = pd.DataFrame(results)
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"VIN API 결과 저장 완료: {output_file}")

In [31]:
fetch_to_csv(VIN_sample, model_sample, "./Data/vin_api_result.csv")

  0%|          | 0/1 [00:00<?, ?it/s]

sierra1500crewcabslt sierra
silverado1500 silverado
silverado1500crew silverado
tundradoublecabsr tundra
sierra2500hdextendedcab sierra
silverado1500double silverado
coloradoextendedcab colorado
corvettegrandsport corvette
wranglerunlimitedsport wrangler
silverado1500regular silverado
coloradocrewcabz71 colorado
tacomaaccesscabpickup tacoma
camarosscoupe2d camaro
tundracrewmaxsr5pickup tundra
rangersupercrewxlpickup ranger
frontiercrewcabpro4x frontier
f150supercabxlpickup4d f150
tacomadoublecabsr5 tacoma
wranglersportsuv2d wrangler
f150supercrewcabxlt f150


100%|██████████| 1/1 [00:02<00:00,  2.34s/it]


PermissionError: [Errno 13] Permission denied: './Data/vin_api_result.csv'